In [ ]:
from ssh_chain_sq import (PHOTONICChain, PeriodicPHOTONICChain, PeriodicChainSW,
                          OpenPHOTONICChain, OpenChainSW,
                          k_grid, compute_zak_phase, real_space_nambu, to_k_space_nambu,
                          sector_permutation, off_block_norm, nambu_metric,
                          bogoliubov_frequencies, paraunitary_defect)
import numpy as np
import matplotlib.pyplot as plt
import thesis_style as ts
ts.use()

# k-space diagonalisation, Zak phase and cavity dressing of the periodic SSH chain

Everything here runs through `PeriodicChainSW` (in `ssh_chain_sq`).  Build one chain,
then call methods -- each resolves its couplings from the instance but takes keyword
overrides (`Omega_C=`, `Chi_C=`, `omega_r=`, `gamma=`) so a sweep never rebuilds the chain.

The full from-first-principles derivation is in `k_space_diag.ipynb`; the class source
(module helpers + the four subclasses) is in `k_space_diag_impl.ipynb`.

In [ ]:
num_cells, Omega_C, Chi_C, Omega_J, Chi_J = 10, 0.95, 0.5, 1.0, 0.0
g_cap_intra = 0.25 * Omega_C * (1 + Chi_C)
g_cap_inter = 0.25 * Omega_C * (1 - Chi_C)
resonant_omega_r = 2 * (g_cap_intra + g_cap_inter)          # cavity on the interband resonance

chain = PeriodicChainSW(N=num_cells, Omega_C=Omega_C, Chi_C=Chi_C,
                        omega_r=resonant_omega_r, gamma=0.03, fock_photon=6,
                        Omega_J=Omega_J, Chi_J=Chi_J, kerr=0.0)
bz_momenta = chain.momentum_grid()
print(f"g_cap_intra = {g_cap_intra:.4f}  >  g_cap_inter = {g_cap_inter:.4f}   "
      f"(g_cap_intra + g_cap_inter = {g_cap_intra + g_cap_inter:.4f} < 1/2, squeeze exists)")
resonant_momentum, _ = chain.interband_resonant_momentum()
print(f"omega_r = 2(g_cap_intra + g_cap_inter) = {resonant_omega_r:.5f}   ->   "
      f"interband-resonant momentum k* = {resonant_momentum:.5f}  ({resonant_momentum/np.pi:.4f} pi)")

## Momentum space and block-diagonalisation by permutation

Real space -> $k$ space is the operator-basis change $b=\mathcal F c$; grouping the Nambu
indices by the unordered pair $\{k,-k\}$ block-diagonalises $\mathcal H^k$ by a pure
permutation ($8\times8$ generic, $4\times4$ at the TRIMs).  `chain.k_space_blocks(k)` returns
the reduced per-sector blocks directly.

In [ ]:
cap_normal, cap_pairing, joseph_normal, joseph_pairing = real_space_nambu(
    num_cells, Omega_C, Chi_C, Omega_J, Chi_J, PBC=True)
H_cap_k, H_joseph_k = to_k_space_nambu(cap_normal, cap_pairing, joseph_normal, joseph_pairing, num_cells)
sector_perm, sector_blocks = sector_permutation(num_cells)
H_cap_sectored    = H_cap_k[np.ix_(sector_perm, sector_perm)]
H_joseph_sectored = H_joseph_k[np.ix_(sector_perm, sector_perm)]
print(f"off-block ||.||  H_cap {off_block_norm(H_cap_sectored, sector_blocks):.1e}   "
      f"H_joseph {off_block_norm(H_joseph_sectored, sector_blocks):.1e}   (=> exactly block diagonal)")

# chain.k_space_blocks reproduces the reduced blocks of the transform (factor-2 convention)
worst = 0.0
for start, size, k_label in sector_blocks:
    sub = list(range(start, start + size // 2))
    A_t = 2 * H_cap_k[np.ix_(sector_perm[sub], sector_perm[sub])]
    worst = max(worst, np.abs(chain.k_space_blocks(k_label)[0] - A_t).max())
print(f"chain.k_space_blocks vs the F^dag H F transform:  max |diff| = {worst:.1e}")

In [ ]:
block_edges = np.cumsum([size for _, size, _ in sector_blocks])[:-1]
for matrix, name, grouped in [
        (H_cap_k,           r"$H_C$  (k space, natural order)", False),
        (H_cap_sectored,    r"$H_C$  (grouped by $\{k,-k\}$)",   True),
        (H_joseph_sectored, r"$H_J$  (grouped by $\{k,-k\}$)",   True)]:
    image = plt.matshow(np.abs(matrix), cmap="thesis_cmap")
    ax = image.axes
    if grouped:
        for edge in block_edges:
            ax.axhline(edge - 0.5, color="w", lw=0.6); ax.axvline(edge - 0.5, color="w", lw=0.6)
    ax.set_title(name, pad=18, fontsize=9); ax.tick_params(labelsize=7)
    image.figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label=r"$|\mathcal{H}^k_{ij}|$")
plt.show()

## The two bands from the analytic (squeeze) diagonalisation

`chain.diagonalize_cap_sector(k)` is the phased sublattice rotation + two-mode squeeze;
`chain.qp_band_frequencies(k)` reads the bands $\sqrt{1\pm2|\delta_{C,k}|}$ straight off it.

In [ ]:
for k in (1.3, float(bz_momenta[3]), 0.0):
    transform, H_cap_diag = chain.diagonalize_cap_sector(k)
    freqs_diag = np.sort(np.diag(H_cap_diag[:4, :4]).real)          # honest 1/2 convention
    freqs_tau3 = bogoliubov_frequencies(chain.sector_nambu(k)[0], 4)
    print(f"k={k:+.3f}:  paraunitary defect {paraunitary_defect(transform, 4):.1e}   "
          f"bands vs eig(tau3 H)  |diff| {np.abs(freqs_diag - freqs_tau3).max():.1e}")

reference_chain = PHOTONICChain(N=num_cells, Omega_C=Omega_C, Chi_C=Chi_C, omega_r=1.0, gamma=0.0,
                                fock_photon=2, Omega_J=Omega_J, Chi_J=Chi_J, kerr=0.0, PBC=True)
freqs_ssh_chain = np.sort((reference_chain._nambu_prefactor()
                           * np.diag(reference_chain.H_C)[:2 * num_cells]).real)
freqs_from_chain = np.sort([w for k in bz_momenta for w in chain.qp_band_frequencies(k)])
print(f"\nchain.qp_band_frequencies vs PHOTONICChain bands:  max |diff| = "
      f"{np.abs(freqs_ssh_chain - freqs_from_chain).max():.1e}")

k_dense = np.linspace(-np.pi, np.pi, 601)
bloch_dense = np.abs(g_cap_intra + g_cap_inter * np.exp(-1j * k_dense))
freq_alpha_dense, freq_beta_dense = np.sqrt(1 + 2 * bloch_dense), np.sqrt(1 - 2 * bloch_dense)
bands_discrete = np.array([chain.qp_band_frequencies(k) for k in bz_momenta])

In [ ]:
fig, ax = plt.subplots(figsize=ts.figsize(1))
ax.plot(k_dense, freq_alpha_dense, color=ts.COLORS["teal"],
        label=r"$\omega_{k,\alpha}=\sqrt{1+2|\delta_{C,k}|}$")
ax.plot(k_dense, freq_beta_dense, color=ts.COLORS["orange"],
        label=r"$\omega_{k,\beta}=\sqrt{1-2|\delta_{C,k}|}$")
ax.plot(np.repeat(bz_momenta, 2), bands_discrete.ravel(), "o", mfc="none", mec="k", ms=7, lw=0,
        label="chain.qp_band_frequencies")
ax.axhline(1.0, color="0.6", lw=0.7, ls=":")
ax.set_xlabel(r"$k$"); ax.set_ylabel("mode frequency")
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
ax.legend(); ts.finalize(ax); plt.show()

## Zak phase of the lower band (Wilson loop)

`chain.zak_phase()` runs the orbital-gauge Wilson loop.  `paired=True` uses the BdG
$\beta$-band Bogoliubov spinor -- the squeeze/pairing leaves the Zak phase unchanged.
Chiral symmetry: $\gamma_{\rm Zak}=0$ for $g_{c1}>g_{c2}$ ($\chi_C>0$), $\pi$ for $\chi_C<0$.

In [ ]:
for positions in [(0.0, 0.0), (-0.25, 0.25), (0.0, 0.5)]:
    z = chain.zak_phase(sublattice_positions=positions)
    print(f"  s=({positions[0]:+.2f},{positions[1]:+.2f})   gamma_Zak = {z/np.pi:+.4f} pi")
print("\n  bare vs BdG-paired, and vs Chi_C:")
for chi in (-0.5, -0.129, 0.129, 0.5):
    zb = chain.zak_phase(Chi_C=chi, num_k_points=240)
    zp = chain.zak_phase(paired=True, Chi_C=chi, num_k_points=240)
    print(f"    Chi_C={chi:+.3f}:  |bare| {abs(zb)/np.pi:.2f} pi   |paired| {abs(zp)/np.pi:.2f} pi   "
          f"| |bare|-|paired| | {abs(abs(zb) - abs(zp)):.1e}")

chi_values = np.linspace(-0.6, 0.6, 121)
zak_vs_chi = np.array([chain.zak_phase(Chi_C=chi, num_k_points=200) for chi in chi_values])

In [ ]:
fig, ax = plt.subplots(figsize=ts.figsize(1.5))
ax.axvspan(-0.6, 0.0, color=ts.COLORS["teal"], alpha=0.10)
ax.axvspan(0.0, 0.6, color=ts.COLORS["gold"], alpha=0.12)
ax.axvline(0.0, color="0.5", lw=0.8, ls=":")
ax.text(-0.3, 0.5, "topological\n$g_{c1}<g_{c2}$", ha="center", va="center", fontsize=8)
ax.text(0.3, 0.5, "trivial\n$g_{c1}>g_{c2}$", ha="center", va="center", fontsize=8)
ax.plot(chi_values, np.abs(zak_vs_chi) / np.pi, color=ts.COLORS["purple"], lw=1.8)
ax.scatter([Chi_C], [np.abs(chain.zak_phase()) / np.pi], s=70, facecolor="none",
           edgecolor="k", zorder=6, label="notebook params")
ax.set_xlabel(r"$\chi_C$"); ax.set_ylabel(r"$|\gamma_{\rm Zak}|\,/\,\pi$   (lower band, $s=\pm1/4$)")
ax.set_yticks([0, 1]); ax.set_ylim(-0.15, 1.2); ax.set_xlim(-0.6, 0.6); ax.legend()
ts.finalize(ax); plt.show()

## Schrieffer-Wolff correction of the cavity coupling (analytic $\alpha/\beta$ gauge)

`chain.sw_correction_qp(k)` eliminates the resonator in the single-excitation subspace and
returns `(qp_freqs, sw_number, sw_pairing)`: `sw_number` is the $k$-preserving
(number-conserving) piece, `sw_pairing` the $k$-changing (resonator-induced pairing) piece.
Both scale as $\gamma^2$.  Cavity on the interband resonance, $\omega_r=2(g_{c1}+g_{c2})$.

In [ ]:
k_scan = np.linspace(0.06, np.pi, 160)
k_scan = k_scan[np.abs(k_scan - resonant_momentum) > 3e-3]
norm_number, norm_pairing = [], []
for k in k_scan:
    _, sw_number, sw_pairing = chain.sw_correction_qp(k, gamma=0.02, omega_r=resonant_omega_r)
    norm_number.append(np.linalg.norm(sw_number)); norm_pairing.append(np.linalg.norm(sw_pairing))
norm_number, norm_pairing = np.array(norm_number), np.array(norm_pairing)

gamma_scan = np.linspace(0.01, 0.04, 16)
probe_k = resonant_momentum + 0.06
gnum, gpair = [], []
for gm in gamma_scan:
    _, sw_number, sw_pairing = chain.sw_correction_qp(probe_k, gamma=gm, omega_r=resonant_omega_r)
    gnum.append(np.linalg.norm(sw_number)); gpair.append(np.linalg.norm(sw_pairing))
gnum, gpair = np.array(gnum), np.array(gpair)
mid = len(gamma_scan) // 2
sn, sp = gnum[mid] / gamma_scan[mid]**2, gpair[mid] / gamma_scan[mid]**2
print(f"||sw_number||  ~ {sn:.2f} gamma^2   ||sw_pairing|| ~ {sp:.2f} gamma^2   "
      f"(ratio {np.mean(gpair / gnum):.3f}, gamma-independent -> 2nd order)")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
axs[0].semilogy(k_scan, norm_number, color=ts.COLORS["teal"], lw=1.8,
                label=r"$\|\delta A\|$  ($k$-preserving)")
axs[0].semilogy(k_scan, norm_pairing, color=ts.COLORS["orange"], lw=1.8,
                label=r"$\|\delta B\|$  ($k$-changing, induced pairing)")
axs[0].axvline(resonant_momentum, color="0.5", lw=0.8, ls=":")
axs[0].set_xlabel(r"$k$"); axs[0].set_ylabel(r"$\|\delta H_{SW}(k)\|$  ($\gamma=0.02$)")
axs[0].set_title(r"$k$-dependence at $\omega_r = 2(g_{c1}+g_{c2})$", fontsize=10); axs[0].legend(fontsize=8)

axs[1].loglog(gamma_scan, gnum, "o", color=ts.COLORS["teal"], ms=5)
axs[1].loglog(gamma_scan, gpair, "s", color=ts.COLORS["orange"], ms=5)
axs[1].loglog(gamma_scan, sn * gamma_scan**2, "-", color="0.5", lw=1, label=r"$\propto\gamma^2$")
axs[1].loglog(gamma_scan, sp * gamma_scan**2, "-", color="0.5", lw=1)
axs[1].set_xlabel(r"$\gamma$"); axs[1].set_ylabel(r"$\|\delta H_{SW}\|$")
axs[1].set_title(rf"$\gamma$-dependence at $k = k^\star + 0.06$", fontsize=10); axs[1].legend(fontsize=8)
ts.finalize(axs); plt.show()

## $H_{\rm SW}$ in the composite (cavity $\otimes$ chain) space

`chain.sw_correction_composite(k)` is the 2nd-order elimination of the chain pairing folded
into $(\text{cavity Fock})\otimes\{\alpha_k,\beta_k,\alpha_{-k},\beta_{-k}\}$ -- the same
physics as `PHOTONICChain.schrieffer_wolff_correction`, in the analytic gauge.  It is
mode-block-diagonal (dispersive $\Delta n_{\rm ph}=0$ + two-photon $\Delta n_{\rm ph}=\pm2$);
the tiny mode-off-diagonal part is the residual $\alpha_k\!\leftrightarrow\!\beta_k$ interband
scattering.  `chain.joseph_in_qp_gauge(k)` shows why $k$ is never jumped: $\tilde A_J$ is
momentum-block-diagonal, $\tilde B_J$ momentum-block-anti-diagonal.

In [ ]:
from matplotlib.colors import LogNorm
photon_cutoff = 6
panels = [(resonant_momentum + 0.05, r"near $k^\star$"),
          (float(bz_momenta[3]), rf"generic $k={bz_momenta[3]:.2f}$")]
fig, axs = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)
for col, (k, title) in enumerate(panels):
    H_SW = chain.sw_correction_composite(k, photon_cutoff=photon_cutoff, gamma=0.03,
                                         omega_r=resonant_omega_r)
    same_mode = (np.arange(photon_cutoff * 4) % 4)[:, None] == (np.arange(photon_cutoff * 4) % 4)[None, :]
    k_preserving = np.where(same_mode, H_SW.real, 0.0)
    vmax = np.abs(k_preserving).max()
    im0 = axs[0, col].imshow(k_preserving, cmap="thesis_cmap_diverging", vmin=-vmax, vmax=vmax)
    fig.colorbar(im0, ax=axs[0, col], fraction=0.046, pad=0.04)
    axs[0, col].set_title(f"{title}: $k$-preserving", fontsize=10)
    k_changing = np.abs(np.where(~same_mode, H_SW, np.nan))
    hi = np.nanmax(k_changing)
    im1 = axs[1, col].imshow(k_changing, cmap="thesis_cmap", norm=LogNorm(hi * 1e-3, hi))
    fig.colorbar(im1, ax=axs[1, col], fraction=0.046, pad=0.04)
    axs[1, col].set_title(f"{title}: $k$-changing (interband), log", fontsize=10)
    for ax in axs[:, col]:
        for b in range(1, photon_cutoff):
            ax.axhline(4 * b - 0.5, color="0.35", lw=0.5); ax.axvline(4 * b - 0.5, color="0.35", lw=0.5)
        ax.set_xticks(np.arange(photon_cutoff) * 4 + 1.5); ax.set_xticklabels(range(photon_cutoff), fontsize=7)
        ax.set_yticks(np.arange(photon_cutoff) * 4 + 1.5); ax.set_yticklabels(range(photon_cutoff), fontsize=7)
        ax.set_xlabel(r"cavity $n_{\rm ph}$", fontsize=8)
fig.suptitle(r"$H_{SW}$  in  (cavity Fock) $\otimes$ ($\alpha_k,\beta_k,\alpha_{-k},\beta_{-k}$),  $\gamma=0.03$",
             fontsize=11)
plt.show()

In [ ]:
mode_labels = [r"$\alpha_k$", r"$\beta_k$", r"$\alpha_{-k}$", r"$\beta_{-k}$",
               r"$\alpha_k^\dagger$", r"$\beta_k^\dagger$", r"$\alpha_{-k}^\dagger$", r"$\beta_{-k}^\dagger$"]
omega_c_values, chi_c_values = [0.10, 0.70], [-0.60, 0.00, 0.60]
fig, axs = plt.subplots(len(omega_c_values), len(chi_c_values), figsize=(13, 8.5), constrained_layout=True)
print("structural norms  (identically 0, for every parameter):")
for row, oc in enumerate(omega_c_values):
    for col, xc in enumerate(chi_c_values):
        joseph_qp = chain.joseph_in_qp_gauge(1.3, Omega_C=oc, Chi_C=xc)
        number_block, pairing_block = joseph_qp[:4, :4], joseph_qp[:4, 4:]
        print(f"  Omega_C={oc:.2f} Chi_C={xc:+.2f}:  "
              f"||A_J[+k,-k]||={np.linalg.norm(number_block[:2, 2:]):.1e}   "
              f"||B_J[+k,+k]||={np.linalg.norm(pairing_block[:2, :2]):.1e}")
        ax = axs[row, col]
        im = ax.imshow(np.abs(joseph_qp), cmap="thesis_cmap")
        ax.axhline(3.5, color="k", lw=1.4); ax.axvline(3.5, color="k", lw=1.4)
        for b in (1.5, 5.5):
            ax.axhline(b, color="w", lw=0.8, ls="--"); ax.axvline(b, color="w", lw=0.8, ls="--")
        ax.set_xticks(range(8)); ax.set_yticks(range(8))
        ax.set_xticklabels(mode_labels if row == len(omega_c_values) - 1 else [], fontsize=6, rotation=90)
        ax.set_yticklabels(mode_labels if col == 0 else [], fontsize=6)
        ax.set_title(rf"$\Omega_C={oc}$,  $\chi_C={xc:+.1f}$", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(r"$|\tilde H_J(k=1.3)|$ in the $H_C$-diagonalising gauge  "
             r"(solid: particle/hole,  dashed: $+k$/$-k$)", fontsize=11)
plt.show()

In [ ]:
def peak_k_changing(Omega_C_val, Chi_C_val, momenta, photon_cutoff=4):
    """max over the BZ of the largest mode-off-diagonal |H_SW| element, at gamma=1
    (H_SW ~ gamma^2 exactly, so this is the gamma^2-normalised value)."""
    qp_mode = np.arange(4 * photon_cutoff) % 4
    off = qp_mode[:, None] != qp_mode[None, :]
    return max(np.abs(chain.sw_correction_composite(
        k, photon_cutoff=photon_cutoff, gamma=1.0, omega_r=Omega_C_val,
        Omega_C=Omega_C_val, Chi_C=Chi_C_val)[off]).max() for k in momenta)

bz_scan   = np.linspace(0.05, np.pi, 20)
chi_grid  = np.linspace(-0.95, 0.95, 25)
omc_grid  = np.linspace(0.01, 0.95, 25)
peak_grid = np.array([[peak_k_changing(oc, xc, bz_scan) for xc in chi_grid] for oc in omc_grid])
# omega_r = Omega_C sits ON the interband resonance, so the raw magnitude is
# regulator-sensitive (the k-grid samples near the pole).  What is robust:
print(f"on Chi_C = 0: max = {peak_grid[:, np.argmin(np.abs(chi_grid))].max():.1e}  "
      f"(vanishes -- band-diagonal pairing forbids alpha_k -> beta_k)")
print(f"argmax at (Chi_C, Omega_C) = ({chi_grid[np.argmax(peak_grid) % 25]:+.2f}, "
      f"{omc_grid[np.argmax(peak_grid) // 25]:.2f})   "
      f"(grows monotonically with |Chi_C| and Omega_C, symmetric in Chi_C)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5), constrained_layout=True)
im = ax.pcolormesh(chi_grid, omc_grid, np.clip(peak_grid, 1e-3, None), cmap="thesis_cmap",
                   shading="gouraud", vmin=1e-3, vmax=10)
ax.set_xlabel(r"$\chi_C$"); ax.set_ylabel(r"$\Omega_C$")
ax.set_title(r"$\max_k\,|H_{SW}^{\,k\text{-chg}}(k)|\,/\,\gamma^2$   "
             r"(0 on $\chi_C=0$; grows with $|\chi_C|$, $\Omega_C$)", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## Cavity-dressed effective Hamiltonian per momentum

`chain.dressed_hamiltonian(k)` = $\omega_r a^\dagger a + \operatorname{diag}(\omega_{k\alpha},\omega_{k\beta})
-\gamma(a+a^\dagger)\otimes M$ with the chain pairing folded in via SW (`include_sw=False`
drops it instead).  `chain.dressed_sector(k)` is the same on the full $\{k,-k\}$ sector --
the $+k\leftrightarrow-k$ coupling is identically zero, so `dressed_hamiltonian` is an exact
block, not a truncation.

In [ ]:
photon_cutoff, gamma_ref, sw_floor = 5, 0.03, 0.05
generic = bz_momenta[np.abs(np.sin(bz_momenta)) > 1e-9]
plus_k  = [n * 4 + s for n in range(photon_cutoff) for s in (0, 1)]
minus_k = [n * 4 + s for n in range(photon_cutoff) for s in (2, 3)]
c_dir = c_sw = c_hsw = 0.0
for k in generic:
    c_dir = max(c_dir, np.linalg.norm(chain.dressed_sector(
        k, photon_cutoff, include_sw=False, gamma=gamma_ref)[np.ix_(plus_k, minus_k)]))
    c_sw = max(c_sw, np.linalg.norm(chain.dressed_sector(
        k, photon_cutoff, include_sw=True, gamma=gamma_ref)[np.ix_(plus_k, minus_k)]))
    c_hsw = max(c_hsw, np.linalg.norm(chain.sw_correction_composite(
        k, photon_cutoff, gamma=gamma_ref)[np.ix_(plus_k, minus_k)]))
print(f"max |+k <-> -k| over the BZ:  direct {c_dir:.1e}   H_SW {c_hsw:.1e}   full {c_sw:.1e}")
print(f"=> every {2*photon_cutoff}-dim momentum block is exact.\n")

block_dim, num_k = 2 * photon_cutoff, len(bz_momenta)
blocks_per_k = [chain.dressed_hamiltonian(k, photon_cutoff, gamma=gamma_ref, denom_floor=sw_floor)
                for k in bz_momenta]
block_diagonal = np.zeros((block_dim * num_k, block_dim * num_k), dtype=complex)
for m, blk in enumerate(blocks_per_k):
    block_diagonal[m*block_dim:(m+1)*block_dim, m*block_dim:(m+1)*block_dim] = blk
perm = np.array([(s * photon_cutoff + n) * num_k + m
                 for m in range(num_k) for n in range(photon_cutoff) for s in (0, 1)])
natural = block_diagonal[np.ix_(np.argsort(perm), np.argsort(perm))]
def offblk(M, b):
    o = M.copy()
    for i in range(0, M.shape[0], b): o[i:i+b, i:i+b] = 0.0
    return np.linalg.norm(o)
print(f"off-(2 photon_cutoff)-block norm:  natural {offblk(natural, block_dim):.2e}  ->  "
      f"momentum-major {offblk(block_diagonal, block_dim):.1e}   ({num_k} blocks of {block_dim})")

In [ ]:
from matplotlib.colors import LogNorm
def strip_diag(M):
    a = np.abs(M).copy(); np.fill_diagonal(a, 0.0); return a
nat_od, bd_od = strip_diag(natural), strip_diag(block_diagonal)
cmax = bd_od.max()
fig, axs = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
for ax, M, title in [(axs[0], nat_od, "natural order  (BZ points interleaved)"),
                     (axs[1], bd_od,  f"momentum-major:  {num_k} blocks of {block_dim}")]:
    im = ax.imshow(np.where(M > 0, M, np.nan), cmap="thesis_cmap", norm=LogNorm(cmax * 1e-4, cmax))
    ax.set_title(title, fontsize=13); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
for e in range(block_dim, block_dim * num_k, block_dim):
    axs[1].axhline(e - 0.5, color="0.4", lw=0.6); axs[1].axvline(e - 0.5, color="0.4", lw=0.6)
fig.suptitle(rf"off-diagonal $|H_{{\rm eff}}|$ over the BZ   "
             rf"($N={num_cells}$, $d_{{\rm ph}}={photon_cutoff}$, $\gamma={gamma_ref}$; every coupling stays in its momentum block)",
             fontsize=13)
plt.show()

In [ ]:
k_fine = np.linspace(-np.pi, np.pi, 361)
E_drop = np.array([np.linalg.eigvalsh(chain.dressed_hamiltonian(k, photon_cutoff, include_sw=False,
                                                                gamma=gamma_ref)) for k in k_fine])
E_sw   = np.array([np.linalg.eigvalsh(chain.dressed_hamiltonian(k, photon_cutoff, gamma=gamma_ref,
                                                                denom_floor=sw_floor)) for k in k_fine])
E_bz   = np.array([np.linalg.eigvalsh(chain.dressed_hamiltonian(k, photon_cutoff, gamma=gamma_ref,
                                                                denom_floor=sw_floor)) for k in bz_momenta])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True, constrained_layout=True)
for ax, E, title in [(axs[0], E_drop, "chain pairing dropped"),
                     (axs[1], E_sw,   "chain pairing folded in via SW")]:
    ax.plot(k_fine, E, color="0.75", lw=1.0)
    ax.plot(k_fine, E[:, 0], color=ts.COLORS["teal"], lw=2.4)
    ax.plot(k_fine, E[:, 1], color=ts.COLORS["orange"], lw=2.4)
    ax.set_xlabel(r"$k$", fontsize=12); ax.set_title(title, fontsize=12)
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r"$-\pi$", "0", r"$\pi$"]); ax.tick_params(labelsize=11)
axs[1].plot(np.repeat(bz_momenta, 2 * photon_cutoff), E_bz.ravel(), "o", mfc="none", mec="k", ms=5, lw=0,
            label="discrete BZ")
axs[1].legend(fontsize=10)
axs[0].set_ylabel(r"$E_n(k)$", fontsize=12)
lo = min(E_drop[:, 0].min(), E_sw[:, 0].min()) - 0.1
axs[0].set_ylim(lo, lo + 6 * resonant_omega_r)
fig.suptitle(rf"per-momentum dressed spectrum  ($d_{{\rm ph}}={photon_cutoff}$, $\gamma={gamma_ref}$, "
             rf"$\omega_r={resonant_omega_r:.3f}$)", fontsize=13)
ts.finalize(axs); plt.show()

## Zak phase of the cavity-dressed second band

`chain.dressed_zak_phase(band)` lifts each eigenvector into the fixed
(sublattice $\otimes$ cavity Fock) basis and runs the Wilson loop.  Band 1 (the lower
polariton of the $|\alpha_k,0\rangle$-$|\beta_k,1\rangle$ anticrossing) picks up
$\gamma_{\rm Zak}=\pi$ once the cavity dresses it -- the cavity frequency is a topological knob.

In [ ]:
for band, label in [(0, "band 0  (|beta_k,0>)"), (1, "band 1  (2nd, lower polariton)"),
                    (2, "band 2  (upper polariton)"), ([1, 2], "bands {1,2}")]:
    print(f"  gamma_Zak[{label:32s}] = {chain.dressed_zak_phase(band, num_k_points=180)/np.pi:+.4f} pi")
print("\ncavity switches the second band topological:")
for gm in (0.0, 0.01, 0.03, 0.05):
    print(f"  gamma={gm:.3f}:  {chain.dressed_zak_phase(1, num_k_points=180, gamma=gm)/np.pi:+.4f} pi")

bloch_half = np.abs(g_cap_intra + g_cap_inter * np.exp(-1j * np.linspace(0, np.pi, 400)))
gap = np.sqrt(1 + 2 * bloch_half) - np.sqrt(1 - 2 * bloch_half)
gap_min, gap_max = float(gap.min()), float(gap.max())
omega_r_scan = np.linspace(0.05, 2.5, 32)
zak1  = np.array([chain.dressed_zak_phase(1,      num_k_points=120, gamma=0.02, omega_r=wr) for wr in omega_r_scan])
zak12 = np.array([chain.dressed_zak_phase([1, 2], num_k_points=120, gamma=0.02, omega_r=wr) for wr in omega_r_scan])
E_omr = np.array([np.linalg.eigvalsh(chain.dressed_hamiltonian(k, 6, gamma=0.02,
                                                               omega_r=0.5 * (gap_min + gap_max)))
                  for k in np.linspace(-np.pi, np.pi, 241)])

In [ ]:
kk = np.linspace(-np.pi, np.pi, 241)
fig, axs = plt.subplots(1, 2, figsize=(14, 5.4), constrained_layout=True)
axs[0].plot(kk, E_omr[:, :6], color="0.7", lw=1.0)
axs[0].plot(kk, E_omr[:, 0], color=ts.COLORS["teal"], lw=1.8, label="band 0")
axs[0].plot(kk, E_omr[:, 1], color=ts.COLORS["orange"], lw=1.8, label="band 1 (2nd)")
axs[0].plot(kk, E_omr[:, 2], color=ts.COLORS["red"], lw=1.2, label="band 2")
axs[0].set_xlabel(r"$k$"); axs[0].set_ylabel(r"$E_n(k)$")
axs[0].set_xticks([-np.pi, 0, np.pi]); axs[0].set_xticklabels([r"$-\pi$", "0", r"$\pi$"])
axs[0].set_ylim(E_omr[:, :4].min() - 0.02, E_omr[:, 3].max() + 0.02)
axs[0].set_title(rf"$\omega_r={0.5*(gap_min+gap_max):.3f}$ (in the gap window)", fontsize=10); axs[0].legend(fontsize=8)

axs[1].axvspan(gap_min, gap_max, color=ts.COLORS["gold"], alpha=0.15, label="interband gap window")
axs[1].plot(omega_r_scan, np.abs(zak1) / np.pi,  "o-",  color=ts.COLORS["orange"], ms=4, label="band 1")
axs[1].plot(omega_r_scan, np.abs(zak12) / np.pi, "s--", color=ts.COLORS["purple"], ms=3, label=r"bands $\{1,2\}$")
axs[1].set_xlabel(r"$\omega_r$"); axs[1].set_ylabel(r"$|\gamma_{\rm Zak}|\,/\,\pi$")
axs[1].set_yticks([0, 1]); axs[1].set_ylim(-0.15, 1.2)
axs[1].set_title("Zak phase vs cavity frequency", fontsize=10); axs[1].legend(fontsize=8)
ts.finalize(axs); plt.show()

## The two bands as a two-level system

$|g\rangle=(0,1,0)=|\alpha_k,0\rangle$, $|e\rangle=(1,0,1)=|\beta_k,1\rangle$ -- degenerate at
$k^\star$, split by the JC anticrossing.  `chain.tls_downfold(H, [0,3])` reduces the dressed
block to the $2\times2$ doublet (diagonalise $P$/$Q$ blocks, rotate the coupling, 2nd-order
shift).  `chain.tls_three_gaps(k)` returns the doublet gap three ways: exact, bare $2\times2$,
and SW-downfolded.

In [ ]:
k0 = float(bz_momenta[np.argmin(np.abs(bz_momenta - resonant_momentum))])
photon_cutoff_tls = 6
H_eff = chain.dressed_hamiltonian(k0, photon_cutoff_tls, gamma=gamma_ref, denom_floor=sw_floor)
triples = [(n, 1, 0) if s == 0 else (n, 0, 1) for n in range(photon_cutoff_tls) for s in (0, 1)]
target = [triples.index((0, 1, 0)), triples.index((1, 0, 1))]
adj = np.abs(H_eff) > 1e-12
reached, queue = set(target), list(target)
while queue:
    for j in np.where(adj[queue.pop()])[0]:
        if j not in reached:
            reached.add(j); queue.append(int(j))
connected = sorted(reached)
print(f"k0 = {k0:.3f}:  connected Fock states {len(connected)} of {2*photon_cutoff_tls} "
      f"({'whole photon ladder' if len(connected) == 2*photon_cutoff_tls else [triples[i] for i in connected]})")
order = target + [i for i in connected if i not in target]
H_reordered = H_eff[np.ix_(order, order)]
labels = [str(triples[i]) for i in order]
H_PP = H_reordered[:2, :2]
print("H_PP (|g>,|e>):\n", np.round(H_PP, 5))
print(f"bare doublet gap = {np.ptp(np.linalg.eigvalsh(H_PP)):.5e}   "
      f"SW-downfolded gap = {np.ptp(np.linalg.eigvalsh(chain.tls_downfold(H_reordered, [0, 1]))):.5e}")

In [ ]:
from matplotlib.colors import LogNorm
od = np.abs(H_reordered).copy(); np.fill_diagonal(od, 0.0)
fig, axs = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
im0 = axs[0].imshow(np.abs(H_reordered), cmap="thesis_cmap")
axs[0].set_title(r"$|H_{\rm eff}(k_0)|$  (diagonal = bare energies)", fontsize=13)
fig.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.03)
im1 = axs[1].imshow(np.where(od > 0, od, np.nan), cmap="thesis_cmap", norm=LogNorm(od.max() * 1e-3, od.max()))
axs[1].set_title("off-diagonal only  (which Fock states are connected)", fontsize=13)
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.03)
for ax in axs:
    ax.axhline(1.5, color="w", lw=1.6); ax.axvline(1.5, color="w", lw=1.6)
    ax.set_xticks(range(len(order))); ax.set_yticks(range(len(order)))
    ax.set_xticklabels(labels, fontsize=9, rotation=90); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel(r"$(n_{\rm res},\,n_\alpha,\,n_\beta)$", fontsize=11)
fig.suptitle(rf"reordered $[\,|g\rangle=(0,1,0),\ |e\rangle=(1,0,1),\ \mathrm{{rest}}\,]$,   $k_0={k0:.3f}$",
             fontsize=13)
plt.show()

In [ ]:
k_gap = np.linspace(0.06, np.pi - 0.02, 221)
gamma_gaps = [0.015, 0.03, 0.05]
gaps_by_gamma = [np.array([chain.tls_three_gaps(k, gamma=gm, omega_r=resonant_omega_r, denom_floor=sw_floor)
                           for k in k_gap]) for gm in gamma_gaps]

gamma_sweep = np.logspace(-2.4, -1.05, 18)
errors = np.array([[abs(b - a), abs(c - a)]
                   for a, b, c in (chain.tls_three_gaps(k0, gamma=gm, omega_r=resonant_omega_r,
                                                        denom_floor=sw_floor) for gm in gamma_sweep)])
slope = lambda e: np.log(e[2] / e[0]) / np.log(gamma_sweep[2] / gamma_sweep[0])
print(f"at k0 (small-gamma slope):  |bare - exact| ~ gamma^{slope(errors[:, 0]):.2f}   "
      f"|SW - exact| ~ gamma^{slope(errors[:, 1]):.2f}")
for oc, xc, fac in [(Omega_C, Chi_C, 1.0), (0.60, 0.50, 1.0), (0.95, 0.80, 1.0),
                    (0.95, 0.50, 0.85), (0.95, 0.50, 1.15)]:
    gi, ge = 0.25 * oc * (1 + xc), 0.25 * oc * (1 - xc)
    wr = fac * 2 * (gi + ge)
    xs = 0.5 * wr * np.sqrt(1 - wr**2 / 4)
    ks = np.arccos(np.clip((xs**2 - gi**2 - ge**2) / (2 * gi * ge), -1, 1))
    a, b, c = chain.tls_three_gaps(ks, Omega_C=oc, Chi_C=xc, omega_r=wr, gamma=gamma_ref)
    print(f"  Omega_C={oc:.2f} Chi_C={xc:+.2f} omega_r={wr:.3f}:  gap {a:.3e}  "
          f"bare err {abs(b-a):.1e}  SW err {abs(c-a):.1e}")

In [ ]:
gmax = 1.05 * max(np.nanmax(g[:, :3]) for g in gaps_by_gamma)
fig, axs = plt.subplots(2, 3, figsize=(16, 8.4), sharex=True, constrained_layout=True)
for col, (gm, g) in enumerate(zip(gamma_gaps, gaps_by_gamma)):
    top, bot = axs[0, col], axs[1, col]
    top.plot(k_gap, g[:, 0], color="k", lw=2.6, label="exact")
    top.plot(k_gap, g[:, 1], color=ts.COLORS["teal"], lw=1.8, ls="--", label=r"bare $H_{PP}$")
    top.plot(k_gap, g[:, 2], color=ts.COLORS["orange"], lw=1.8, ls=":", label="SW doublet")
    top.set_ylim(0, gmax); top.set_title(rf"$\gamma={gm}$", fontsize=12)
    bot.axhline(0, color="0.6", lw=0.8)
    bot.plot(k_gap, g[:, 1] - g[:, 0], color=ts.COLORS["teal"], lw=1.8, ls="--", label=r"bare $-$ exact")
    bot.plot(k_gap, g[:, 2] - g[:, 0], color=ts.COLORS["orange"], lw=1.8, ls=":", label=r"SW $-$ exact")
    bot.set_xlabel(r"$k$", fontsize=12)
    for ax in (top, bot):
        ax.axvline(resonant_momentum, color="0.6", lw=0.9, ls=":")
        ax.set_xticks([0, np.pi/2, np.pi]); ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$"]); ax.tick_params(labelsize=11)
axs[0, 0].set_ylabel(r"doublet gap  $|E_e - E_g|$", fontsize=12)
axs[1, 0].set_ylabel("gap error", fontsize=12)
axs[0, 0].legend(fontsize=10, loc="upper left"); axs[1, 0].legend(fontsize=10, loc="lower left")
fig.suptitle(rf"$|g\rangle=(0,1,0)$, $|e\rangle=(1,0,1)$ as a TLS  "
             rf"($\omega_r={resonant_omega_r:.3f}$, $\Omega_C={Omega_C}$, $\chi_C={Chi_C}$)", fontsize=13)
ts.finalize(axs); plt.show()

fig, ax = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
ax.loglog(gamma_sweep, errors[:, 0], "o-", color=ts.COLORS["teal"], ms=6, lw=1.6,
          label=r"$|\Delta_{\rm bare} - \Delta_{\rm exact}|$")
ax.loglog(gamma_sweep, errors[:, 1], "s-", color=ts.COLORS["orange"], ms=6, lw=1.6,
          label=r"$|\Delta_{\rm SW} - \Delta_{\rm exact}|$")
ax.loglog(gamma_sweep, errors[0, 0] * (gamma_sweep / gamma_sweep[0])**2, "--", color="0.55", lw=1.2,
          label=r"$\propto\gamma^2$")
ax.loglog(gamma_sweep, errors[0, 1] * (gamma_sweep / gamma_sweep[0])**4, ":", color="0.55", lw=1.2,
          label=r"$\propto\gamma^4$")
ax.set_xlabel(r"$\gamma$"); ax.set_ylabel(r"doublet-gap error at $k^\star$")
ax.legend(fontsize=11); ts.finalize(ax); plt.show()

## Analytic coefficients of the transformed $H_J$ (symbolic)

Redo the $H_C$-diagonalising transform $T = T_{\rm rot}\,T_{\rm sq}$ (phased sublattice
rotation into $\alpha_k/\beta_k$, then the two-mode squeeze) with `sympy`, apply it to
$H_J$, and read off closed forms for every entry of $\tilde{\mathcal A}_J$ (number,
QP gauge) and $\tilde{\mathcal B}_J$ (pairing).  Same derivation as the Mathematica
notebook `hj_analytic_coefficients.nb`.  Fundamental symbols: $ad=|\delta_{C,k}|$
$(0<ad<\tfrac12)$, $\varphi=\arg\delta_{C,k}$, $\delta_J=\delta_{J,k}$ (generic complex),
$g_J=g_{J1}+g_{J2}$; the bands are $w_\alpha^2 = 1+2\,ad$, $w_\beta^2 = 1-2\,ad$.

In [ ]:
import sympy as sp
from sympy import I, exp, sqrt, Rational, conjugate, simplify, symbols, Matrix, zeros, diag

ad = symbols('ad', positive=True)               # |delta_C,k| ,  0 < ad < 1/2
ph = symbols('varphi', real=True)               # arg delta_C,k
gJ = symbols('g_J', real=True)                   # g_J1 + g_J2
dJ = symbols('delta_J'); dJc = conjugate(dJ)     # delta_J,k  (generic complex)
w_alpha, w_beta = symbols('w_alpha w_beta', positive=True)   # w_alpha^2 = 1+2ad, w_beta^2 = 1-2ad
band_sub = {w_alpha: sqrt(1 + 2*ad), w_beta: sqrt(1 - 2*ad)}
dCk, dCkc = ad*exp(I*ph), ad*exp(-I*ph)

# --- the 8x8 sector Nambu blocks (honest 1/2 convention, transcribed from sector_nambu) ---
def sector_from_bloch(on_site, bloch, bloch_conj):
    return diag(Matrix([[on_site, bloch], [bloch_conj, on_site]]),
                Matrix([[on_site, bloch_conj], [bloch, on_site]]))

cap_normal  = Rational(1, 2) * sector_from_bloch(1, dCk, dCkc)
cap_pairing = zeros(4)
cap_pairing[0, 3] = cap_pairing[3, 0] = -dCk/2
cap_pairing[1, 2] = cap_pairing[2, 1] = -dCkc/2
HC_sym = Matrix(sp.BlockMatrix([[cap_normal, cap_pairing],
                                [cap_pairing.conjugate(), cap_normal.conjugate()]]))

joseph_normal  = Rational(1, 2) * sector_from_bloch(gJ, -dJ, -dJc)
joseph_pairing = zeros(4)
joseph_pairing[0, 2] = joseph_pairing[2, 0] = joseph_pairing[1, 3] = joseph_pairing[3, 1] = -gJ/2
joseph_pairing[0, 3] = joseph_pairing[3, 0] = -dJ/2
joseph_pairing[1, 2] = joseph_pairing[2, 1] = -dJc/2
HJ_sym = Matrix(sp.BlockMatrix([[joseph_normal, joseph_pairing],
                                [joseph_pairing.conjugate(), joseph_normal.conjugate()]]))

# --- T = T_rot . T_sq ---
eh, ehc = exp(I*ph/2), exp(-I*ph/2)
rot4 = diag((1/sqrt(2))*Matrix([[ehc, eh], [ehc, -eh]]),      # rows = alpha_k, beta_k
            (1/sqrt(2))*Matrix([[eh, ehc], [eh, -ehc]]))       # rows = alpha_-k, beta_-k
rot_inv = simplify(rot4.inv())
T_rot = Matrix(sp.BlockMatrix([[rot_inv, zeros(4)], [zeros(4), rot_inv.conjugate()]]))

chA, shA = (sqrt(w_alpha) + 1/sqrt(w_alpha))/2, (sqrt(w_alpha) - 1/sqrt(w_alpha))/2
chB, shB = (sqrt(w_beta)  + 1/sqrt(w_beta))/2,  (sqrt(w_beta)  - 1/sqrt(w_beta))/2
U4 = diag(chA, chB, chA, chB)
V4 = zeros(4); V4[0, 2] = V4[2, 0] = shA; V4[1, 3] = V4[3, 1] = shB
T_sq = Matrix(sp.BlockMatrix([[U4, V4], [V4, U4]]))

T_sym = sp.expand(T_rot * T_sq)
tau3  = diag(1, 1, 1, 1, -1, -1, -1, -1)
print("built HC_sym, HJ_sym (8x8) and the paraunitary transform T_sym")

In [ ]:
# T is paraunitary
print("T^dag tau3 T - tau3 is the zero matrix:",
      simplify(T_sym.conjugate().T @ tau3 @ T_sym - tau3).is_zero_matrix)

# T diagonalises H_C: check off-diagonal numerically, then read the bands
HCd_sym = T_sym.conjugate().T @ HC_sym @ T_sym
_pt = {ad: 0.31, ph: 0.4, gJ: 0.5, dJ: 0.6 + 0.3j,
       w_alpha: (1 + 2*0.31)**0.5, w_beta: (1 - 2*0.31)**0.5}
HCd_num = np.array(HCd_sym.subs(_pt).evalf(), dtype=complex)
print("H_Cd off-diagonal magnitude at a sample point:",
      np.abs(HCd_num - np.diag(np.diag(HCd_num))).max())
print("physical bands  2*diag(H_Cd)[:2] =",
      [sp.simplify((2*HCd_sym[i, i]).subs(band_sub)) for i in range(2)],
      "  ( = sqrt(1 +/- 2 ad) )")

In [ ]:
HJd_sym = T_sym.conjugate().T @ HJ_sym @ T_sym
A_J_sym = sp.Matrix(2 * HJd_sym[:4, :4])      # physical number block   \tilde A_J  (alpha_k, beta_k, alpha_-k, beta_-k)
B_J_sym = sp.Matrix(2 * HJd_sym[:4, 4:])      # physical pairing block  \tilde B_J

cf = lambda e: sp.factor(sp.simplify(e))
coeffs = {
    "A_J[alpha_k, alpha_k]"                       : cf(A_J_sym[0, 0]),
    "A_J[beta_k,  beta_k]"                        : cf(A_J_sym[1, 1]),
    "A_J[alpha_k, beta_k]   (interband, same k)"  : cf(A_J_sym[0, 1]),
    "A_J[alpha_k, {alpha,beta}_-k]  (should be 0)"       : [sp.simplify(A_J_sym[0, 2]), sp.simplify(A_J_sym[0, 3])],
    "B_J[alpha_k, alpha_-k] (zero-momentum pair)" : cf(B_J_sym[0, 2]),
    "B_J[alpha_k, beta_-k]  (zero-momentum pair)" : cf(B_J_sym[0, 3]),
    "B_J[beta_k,  beta_-k]  (zero-momentum pair)" : cf(B_J_sym[1, 3]),
    "B_J[alpha_k, {alpha_k,beta_k}]  (should be 0)"      : [sp.simplify(B_J_sym[0, 0]), sp.simplify(B_J_sym[0, 1])],
}
for name, val in coeffs.items():
    print(f"{name:44s} = {val}")
print("\nA_J[alpha_k, beta_k] == B_J[alpha_k, beta_-k] :",
      sp.simplify(A_J_sym[0, 1] - B_J_sym[0, 3]) == 0)
print("=> A_J is momentum-block-diagonal, B_J momentum-block-anti-diagonal (Step 9b).")

In [ ]:
# --- write the coefficients down (rendered from A_J_sym / B_J_sym) -----------
from IPython.display import display, Math
import sympy as sp

def _show(lhs, expr):
    display(Math(lhs + " = " + sp.latex(sp.simplify(expr))))
    display(Math(r"\qquad = " + sp.latex(sp.simplify(sp.simplify(expr).subs(band_sub)))
                 + r"\quad\text{(explicit in } ad=|\delta_{C,k}|\text{)}"))

print("physical  \\tilde H_J  in the QP gauge   "
      "( w_alpha^2 = 1 + 2 ad ,  w_beta^2 = 1 - 2 ad ):\n")
_show(r"\tilde{\mathcal{A}}_J[\alpha_k,\alpha_k]",   A_J_sym[0, 0])
_show(r"\tilde{\mathcal{A}}_J[\beta_k,\beta_k]",     A_J_sym[1, 1])
_show(r"\tilde{\mathcal{A}}_J[\alpha_k,\beta_k]",    A_J_sym[0, 1])
_show(r"\tilde{\mathcal{B}}_J[\alpha_k,\alpha_{-k}]", B_J_sym[0, 2])
_show(r"\tilde{\mathcal{B}}_J[\alpha_k,\beta_{-k}]",  B_J_sym[0, 3])
_show(r"\tilde{\mathcal{B}}_J[\beta_k,\beta_{-k}]",   B_J_sym[1, 3])
display(Math(r"\tilde{\mathcal{A}}_J[\alpha_k,\{\alpha,\beta\}_{-k}] = 0"
             r"\qquad\tilde{\mathcal{B}}_J[\alpha_k,\{\alpha_k,\beta_k\}] = 0"))

### The closed forms

With $\delta_{J,k}=g_{J1}+g_{J2}e^{-ik}$, $\varphi=\arg\delta_{C,k}$, $g_J=g_{J1}+g_{J2}$
and the bands $w_{\alpha,\beta}=\sqrt{1\pm 2|\delta_{C,k}|}$, the transform
$T=T_{\rm rot}T_{\rm sq}$ takes the Josephson block to (physical convention, QP basis
$\alpha_k,\beta_k,\alpha_{-k},\beta_{-k}$):

$$\tilde{\mathcal A}_J[\alpha_k,\alpha_k]
 =-\tfrac{w_\alpha}{2}e^{-i\varphi}\delta_{J,k}+\tfrac{g_J}{w_\alpha}
   -\tfrac{w_\alpha}{2}e^{i\varphi}\bar\delta_{J,k},\qquad
  \tilde{\mathcal A}_J[\beta_k,\beta_k]
 =\tfrac{w_\beta}{2}e^{-i\varphi}\delta_{J,k}+\tfrac{g_J}{w_\beta}
   +\tfrac{w_\beta}{2}e^{i\varphi}\bar\delta_{J,k}$$

$$\boxed{\ \tilde{\mathcal A}_J[\alpha_k,\beta_k]
        =\tilde{\mathcal B}_J[\alpha_k,\beta_{-k}]
        =\frac{(1-4|\delta_{C,k}|^2)^{1/4}}{2}\,e^{-i\varphi}
         \big(\delta_{J,k}-e^{2i\varphi}\,\bar\delta_{J,k}\big)\ }$$

$$\tilde{\mathcal B}_J[\alpha_k,\alpha_{-k}]
 =-\tfrac{w_\alpha}{2}e^{-i\varphi}\delta_{J,k}-\tfrac{g_J}{w_\alpha}
   -\tfrac{w_\alpha}{2}e^{i\varphi}\bar\delta_{J,k},\qquad
  \tilde{\mathcal B}_J[\beta_k,\beta_{-k}]
 =\tfrac{w_\beta}{2}e^{-i\varphi}\delta_{J,k}-\tfrac{g_J}{w_\beta}
   +\tfrac{w_\beta}{2}e^{i\varphi}\bar\delta_{J,k}$$

$$\tilde{\mathcal A}_J[\alpha_k,\alpha_{-k}]=\tilde{\mathcal A}_J[\alpha_k,\beta_{-k}]=0,
  \qquad
  \tilde{\mathcal B}_J[\alpha_k,\alpha_k]=\tilde{\mathcal B}_J[\alpha_k,\beta_k]=0
  \qquad(\text{Step 9b}).$$

**Readout.**

* The band-diagonal number shift $\tilde{\mathcal A}_J[\gamma_k,\gamma_k]$ and the
  same-mode induced pairing $\tilde{\mathcal B}_J[\gamma_k,\gamma_{-k}]$ differ **only**
  by the sign of the $g_J/w_\gamma$ term.
* The interband number coupling $\tilde{\mathcal A}_J[\alpha_k,\beta_k]$ and the
  $\alpha_k$–$\beta_{-k}$ zero-momentum pair $\tilde{\mathcal B}_J[\alpha_k,\beta_{-k}]$
  carry the **identical** coefficient, $\propto\ \delta_{J,k}-e^{2i\varphi}\bar\delta_{J,k}$.
* That factor vanishes iff $\arg\delta_{J,k}=\varphi=\arg\delta_{C,k}$ (Josephson and
  capacitive Bloch phases aligned) — in particular on $\chi_C=0$, where it is exactly
  the trench seen in the interband ($k$-changing) sweep.

In [ ]:
# --- cross-check the closed forms against the implemented procedure ---
test_points = [                       # (k, gc1, gc2, gJ1, gJ2),  gc1 + gc2 < 1/2
    (0.70, 0.18, 0.11, 1.00, -0.50),
    (1.30, 0.24, 0.16, 1.00,  0.00),
    (2.10, 0.20, 0.19, 0.80, -0.30),
    (0.35, 0.34, 0.05, 1.20, -0.70),
    (3.05, 0.15, 0.14, 1.00, -0.20),   # near k = pi (TRIM)
    (0.90, 0.11, 0.30, 0.60, -0.90),   # topological side  gc1 < gc2
    (1.75, 0.30, 0.09, 0.50, -0.50),
    (0.05, 0.20, 0.20, 0.90, -0.10),   # near k = 0 (TRIM),  chi_C = 0
]
lam_A = sp.lambdify((ad, ph, dJ, gJ, w_alpha, w_beta), A_J_sym, "numpy")
lam_B = sp.lambdify((ad, ph, dJ, gJ, w_alpha, w_beta), B_J_sym, "numpy")

print(f"{'k':>6} {'gc1':>6} {'gc2':>6} {'gJ1':>5} {'gJ2':>6}   {'max|d A_J|':>11}  {'max|d B_J|':>11}   side")
for (k, gc1, gc2, gJ1, gJ2) in test_points:
    OmJ, ChiJ = (gJ1 - gJ2)/2, (gJ1 + gJ2)/(gJ1 - gJ2)
    OC, XC = 2*(gc1 + gc2), (gc1 - gc2)/(gc1 + gc2)
    HC_n, HJ_n = chain.sector_nambu(k, Omega_C=OC, Chi_C=XC, Omega_J=OmJ, Chi_J=ChiJ)
    T_n, _ = chain.diagonalize_cap_sector(k, Omega_C=OC, Chi_C=XC, Omega_J=OmJ, Chi_J=ChiJ)
    HJd_n = T_n.conj().T @ HJ_n @ T_n 
    adv = abs(gc1 + gc2*np.exp(-1j*k)); phv = np.angle(gc1 + gc2*np.exp(-1j*k))
    dJv = gJ1 + gJ2*np.exp(-1j*k); w = (np.sqrt(1 + 2*adv), np.sqrt(1 - 2*adv))
    dA = np.abs(lam_A(adv, phv, dJv, gJ1 + gJ2, *w) - 2*HJd_n[:4, :4]).max()
    dB = np.abs(lam_B(adv, phv, dJv, gJ1 + gJ2, *w) - 2*HJd_n[:4, 4:]).max()
    print(f"{k:6.2f} {gc1:6.2f} {gc2:6.2f} {gJ1:5.1f} {gJ2:6.1f}   {dA:11.2e}  {dB:11.2e}   "
          f"{'trivial' if gc1 > gc2 else 'topological'}")
print("\nall ~1e-15 -> the closed forms match PeriodicChainSW across the BZ and both sides.")